# 📘 Notebook 13: Sequence-to-Sequence Models and Translation

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module E: Neural Networks for Text · Notebook 2 of 3 in this module · (13 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why translation needs a different architecture from tagging or classification
- Describe the **encoder-decoder** (sequence-to-sequence) model and the role of each half
- Explain **teacher forcing** and the difference between training and **decoding**
- Build and train a sequence-to-sequence model in PyTorch and measure its accuracy
- Demonstrate the **bottleneck** of the basic model with an experiment, and say what removes it
- Compute the **BLEU** score by hand, and discuss what it does and does not measure

> **Prerequisites:** Notebook 12 (LSTMs, the training loop for a language model). The idea of attention is introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 15. This notebook shows the problem that attention was invented to solve.
>
> 🔭 **Why this notebook matters:** many of the most useful language tasks take a sequence in and give a **different** sequence out: translation, summarisation, answering a question, correcting a sentence, turning a question into a database query. In 2014 a single simple architecture, two recurrent networks joined end to end, turned out to handle all of them. It replaced, within about two years, machine translation systems that had taken decades to engineer. Its one serious weakness led directly to attention and the Transformer.

> ℹ️ **Setup note.** Training in this notebook takes about two minutes on an ordinary processor.

In [ ]:
import datetime
import math
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. A different kind of problem

### Intuition first
Compare three tasks.

- **Classification** (Notebook 09): a whole text goes in, **one label** comes out.
- **Tagging** (Notebook 06): a sequence goes in, and **one label per word** comes out. Input and output have the same length and line up position by position.
- **Translation**: *"I do not like green tea"* becomes *"Je n'aime pas le thé vert"*.

In translation the two sequences have **different lengths**. The words do not line up one to one: *do not* becomes *ne ... pas*, wrapped around the verb. And the order changes: *green tea* becomes *thé vert*, "tea green". You cannot translate word by word, and you cannot begin to write the output until you have understood the input.

A human interpreter listens to the whole sentence, forms an understanding of it, and then speaks. The sequence-to-sequence model copies that plan literally.

### Formal definition
A **sequence-to-sequence (seq2seq)** model consists of two networks:

- The **encoder** reads the input sequence $x_1 \dots x_n$ and compresses it into a fixed-size **context vector**. With an LSTM encoder, the context is its final hidden and cell state.
- The **decoder** is a language model, exactly like the one in Notebook 12, with one difference: its memory starts from the encoder's context and not from zeros. It then generates the output $y_1 \dots y_m$ one item at a time:

$$P(y_1 \dots y_m \mid x_1 \dots x_n) = \prod_{t=1}^{m} P(y_t \mid y_1 \dots y_{t-1}, \text{context})$$

So a translator is a language model that has been **told what to talk about**.

## 2. A small translation task

Training a real translator needs millions of sentence pairs and hours of computing. To study the architecture we want a task that has the essential features of translation and that trains in under a minute. Here is one: converting dates written by people into the international standard format.

| Input | Output |
|---|---|
| `3 march 2021` | `2021-03-03` |
| `march 3, 2021` | `2021-03-03` |
| `03/03/2021` | `2021-03-03` |
| `3 mar 2021` | `2021-03-03` |

It is a real translation problem in miniature. Input and output have different lengths. The order of the parts changes: the year moves from the end to the front. A word must be turned into a number (*march* into `03`). And a single digit must sometimes be expanded (`3` into `03`). We will work with characters.

In [ ]:
MONTHS = ["january", "february", "march", "april", "may", "june",
          "july", "august", "september", "october", "november", "december"]

def random_date_pair():
    date = datetime.date(1950, 1, 1) + datetime.timedelta(days=random.randint(0, 36500))
    month = MONTHS[date.month - 1]
    style = random.randint(0, 3)
    if style == 0:
        written = f"{date.day} {month} {date.year}"
    elif style == 1:
        written = f"{month} {date.day}, {date.year}"
    elif style == 2:
        written = f"{date.day:02d}/{date.month:02d}/{date.year}"
    else:
        written = f"{date.day} {month[:3]} {date.year}"
    return written, date.isoformat()

random.seed(0)
for i in range(6):
    written, standard = random_date_pair()
    print(f"{written:<20} -> {standard}")

A hundred years of dates in four styles gives about 146,000 different inputs. We generate a fresh random batch at every training step, so the model sees only a part of them, and at the end we test it on dates drawn independently.

### From characters to tensors
Each character becomes a number. Three special symbols are needed:

- `<pad>` fills short sequences so that all the sequences in a batch have the same length,
- `<s>` tells the decoder to start writing,
- `</s>` is what the decoder writes when it has finished.

In [ ]:
PAD, START, END = 0, 1, 2
INPUT_LENGTH = 20                     # every input is padded on the left to this length

def build_vocabularies(make_example):
    examples = [make_example() for i in range(3000)]
    input_characters = ["<pad>"] + sorted(set("".join(source for source, target in examples)))
    output_characters = ["<pad>", "<s>", "</s>"] + sorted(set("".join(target for source, target in examples)))
    return input_characters, output_characters

def encode_input(text, input_characters):
    ids = [input_characters.index(character) for character in text if character in input_characters]
    return [PAD] * (INPUT_LENGTH - len(ids)) + ids

def make_batch(examples, input_characters, output_characters):
    longest = max(len(target) for source, target in examples)
    sources, decoder_inputs, decoder_targets = [], [], []
    for source, target in examples:
        ids = [output_characters.index(character) for character in target]
        padding = [PAD] * (longest - len(ids))
        sources.append(encode_input(source, input_characters))
        decoder_inputs.append([START] + ids + padding)        # what the decoder is given
        decoder_targets.append(ids + [END] + padding)         # what it must predict
    return torch.tensor(sources), torch.tensor(decoder_inputs), torch.tensor(decoder_targets)

date_input_characters, date_output_characters = build_vocabularies(random_date_pair)
print("Input characters: ", "".join(date_input_characters[1:]))
print("Output characters:", "".join(date_output_characters[3:]))

## 3. Teacher forcing

### Intuition first
How do we train the decoder? At each step it must predict the next output character. But the next character depends on the ones before it, and early in training the decoder's own output is rubbish. If it had to continue from its own mistakes, it would get lost after the first character and learn very slowly.

So during training we do not feed the decoder its own output. We feed it the **correct** previous characters, whatever it predicted. This is called **teacher forcing**. It is like a pupil taking dictation whose teacher corrects every word before the pupil goes on to the next one.

In practice this means the decoder's input is the target sequence **shifted right by one**, with `<s>` in front. It is the same trick as in Notebook 12, where the target was the input shifted by one.

In [ ]:
sources, decoder_inputs, decoder_targets = make_batch([("3 march 2021", "2021-03-03")], date_input_characters, date_output_characters)

def show(ids, names):
    return " ".join(names[i] for i in ids.tolist())

print("Encoder input:  ", show(sources[0], date_input_characters))
print("Decoder input:  ", show(decoder_inputs[0], date_output_characters))
print("Decoder target: ", show(decoder_targets[0], date_output_characters))

Read the last two lines as pairs, one column at a time. Given `<s>` the decoder must produce `2`. Given `<s> 2` it must produce `0`. At the end, given the whole date, it must produce `</s>`.

## 4. The model

Two LSTMs and three small layers. Notice the single line that connects the two halves: the decoder LSTM is started with the encoder's final `state`.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, input_vocabulary, output_vocabulary, embedding_size=32, hidden_size=128):
        super().__init__()
        self.input_embedding = nn.Embedding(input_vocabulary, embedding_size, padding_idx=PAD)
        self.output_embedding = nn.Embedding(output_vocabulary, embedding_size, padding_idx=PAD)
        self.encoder = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_vocabulary)

    def encode(self, sources):
        outputs, state = self.encoder(self.input_embedding(sources))
        return state                                          # the context: final hidden and cell state

    def forward(self, sources, decoder_inputs):
        state = self.encode(sources)
        hidden, state = self.decoder(self.output_embedding(decoder_inputs), state)   # started from the context
        return self.output(hidden)

The training loop is the one from Notebook 12. The only new detail is `ignore_index=PAD`, which tells the loss not to count the padding positions.

In [ ]:
def train_seq2seq(make_example, steps, hidden_size=128, report_every=100):
    random.seed(0)
    torch.manual_seed(0)
    input_characters, output_characters = build_vocabularies(make_example)
    model = Seq2Seq(len(input_characters), len(output_characters), hidden_size=hidden_size)
    optimiser = torch.optim.Adam(model.parameters(), lr=3e-3)

    for step in range(1, steps + 1):
        batch = [make_example() for i in range(128)]
        sources, decoder_inputs, decoder_targets = make_batch(batch, input_characters, output_characters)
        scores = model(sources, decoder_inputs)
        loss = F.cross_entropy(scores.reshape(-1, len(output_characters)), decoder_targets.reshape(-1), ignore_index=PAD)
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        if step % report_every == 0:
            print(f"step {step:>4}  loss {loss.item():.4f}")
    return model, input_characters, output_characters

date_system = train_seq2seq(random_date_pair, steps=700)

### Decoding
At training time the decoder was handed the correct answer one step late. When we actually use the model there is no correct answer to hand over. The decoder must run on its own:

1. encode the input to get the context;
2. give the decoder `<s>`;
3. take its most probable next character, and feed that character back in as the next input;
4. stop when it produces `</s>`.

Choosing the single most probable item at every step is called **greedy decoding**.

In [ ]:
def translate(system, text, max_length=30):
    model, input_characters, output_characters = system
    model.eval()
    written = []
    with torch.no_grad():
        state = model.encode(torch.tensor([encode_input(text, input_characters)]))
        token = START
        for i in range(max_length):
            hidden, state = model.decoder(model.output_embedding(torch.tensor([[token]])), state)
            token = model.output(hidden[0, -1]).argmax().item()
            if token == END:
                break
            written.append(output_characters[token])
    model.train()
    return "".join(written)

for text in ["3 march 2021", "march 3, 2021", "03/03/2021", "25 dec 1999", "february 29, 2024"]:
    print(f"{text:<20} -> {translate(date_system, text)}")

And the measurement, on 1,000 dates drawn independently of training. An output counts as correct only if **every** character is right.

In [ ]:
random.seed(12345)
test_pairs = [random_date_pair() for i in range(1000)]

correct = sum(1 for written, standard in test_pairs if translate(date_system, written) == standard)
print(f"Exact-match accuracy: {correct / len(test_pairs):.1%}")

Consider what the model has worked out, from examples alone, in under a minute. It reads four different layouts. It has learned that *march* and *mar* and `03` are the same month. It moves the year to the front, pads single digits with a zero, and knows when to stop. Nobody wrote a rule for any of it. Had we written this converter by hand, in the style of Notebook 02, it would have been a page of regular expressions, and a fifth date format would have meant more code. Here, a fifth format means more examples.

> ⚠️ **Common pitfalls**
>
> - Using teacher forcing at decoding time. There is no teacher. The decoder must consume its own output.
> - Forgetting the end symbol. Without `</s>` in the targets the decoder never learns to stop, and writes until the length limit.
> - Treating padding inconsistently. Here every input is padded on the left to the same length, in training and in use alike. If the two differ, the encoder sees inputs unlike any it was trained on, and accuracy collapses without any error message.

## 5. The bottleneck, measured

### Intuition first
The encoder must squeeze **everything** about the input into one vector of fixed size, and the decoder sees nothing else. For a date of twenty characters, 128 numbers are plenty. But the vector has the same size whether the input is a short date or a sentence of fifty words. It is like summarising any book, short or long, on a single index card, and then asking someone to rewrite the book from the card.

Let us measure it with a task that has no shortcut: **reverse a random string of letters**. Every character matters, and there is no pattern to exploit. We train on strings of 3 to 20 letters and then test each length separately.

In [ ]:
def random_reverse_pair(shortest=3, longest=20):
    length = random.randint(shortest, longest)
    letters = "".join(random.choice("abcdefghij") for i in range(length))
    return letters, letters[::-1]

random.seed(1)
print([random_reverse_pair() for i in range(3)])

reverse_system = train_seq2seq(random_reverse_pair, steps=1200, report_every=200)

In [ ]:
random.seed(54321)
print(f"{'length':>6} {'exact-match accuracy':>22}")
for length in [4, 8, 12, 16, 20]:
    pairs = [random_reverse_pair(length, length) for i in range(200)]
    correct = sum(1 for letters, backwards in pairs if translate(reverse_system, letters) == backwards)
    print(f"{length:>6} {correct / len(pairs):>21.0%}")

Short strings are reversed perfectly. As the strings get longer, accuracy falls, and for the longest ones the model fails often. Nothing is wrong with the training. The longest strings simply carry more information than the fixed-size context can hold reliably.

The same effect was observed in real translation systems of this design: quality was good on short sentences and fell steadily on long ones.

### The way out: attention
The cure is to stop relying on a single summary. Keep **all** the encoder's hidden states, one per input position, and let the decoder **look back** at them at every step, focusing on the positions most relevant to what it is about to write. When it is writing the month, it looks at the part of the input that names the month.

That mechanism is **attention**. It is explained in Notebook 15 of *From Python to Deep Learning & Fine-Tuning*, and Notebook 16 there shows the next step in the reasoning: once every position can look at every other position directly, the recurrent networks are no longer needed at all. The result is the **Transformer**, the architecture of every large language model in use today. The encoder-decoder idea of this notebook survives inside it unchanged. Only the internals of the two halves were replaced.

## 6. Evaluating translation: the BLEU score

### Intuition first
For dates, an output is right or wrong. For real translation there are many correct outputs. *"The cat is sitting on the mat"*, *"The cat sits on the mat"* and *"A cat is seated upon the mat"* are all good translations of the same sentence. Exact match would mark two of them as failures.

The best judges are people, but people are slow and expensive, and developers need to compare systems every day. The practical compromise: compare the machine's output with one or more **reference translations** made by people, and give credit for every word, and every short sequence of words, that the output shares with a reference.

### Formal definition
**BLEU** (Papineni et al., 2002) combines two ingredients.

**1. Modified n-gram precision.** For $n = 1, 2, 3, 4$: of all the n-grams in the candidate translation, what share also occur in the reference? "Modified" means each n-gram of the reference can be matched only as many times as it appears there. This is called **clipping**, and it stops a candidate from scoring by repeating one correct word.

**2. Brevity penalty.** Precision alone rewards saying very little: the two-word output *"the cat"* has perfect precision. So candidates shorter than the reference are penalised:

$$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{\,1 - r/c} & \text{if } c \le r \end{cases}$$

where $c$ is the length of the candidate and $r$ that of the reference. The score is the brevity penalty times the geometric mean of the precisions:

$$\text{BLEU} = \text{BP} \cdot \exp\Big(\frac{1}{N} \sum_{n=1}^{N} \log p_n\Big)$$

It lies between 0 and 1, and is often quoted multiplied by 100.

In [ ]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def modified_precision(candidate, reference, n):
    candidate_counts = ngram_counts(candidate, n)
    reference_counts = ngram_counts(reference, n)
    matched = 0
    for ngram, count in candidate_counts.items():
        matched += min(count, reference_counts[ngram])          # clipping
    return matched / max(1, sum(candidate_counts.values()))

def bleu(candidate, reference, max_n=4):
    candidate, reference = candidate.split(), reference.split()
    precisions = [modified_precision(candidate, reference, n) for n in range(1, max_n + 1)]
    if min(precisions) == 0:
        return 0.0
    geometric_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    if len(candidate) > len(reference):
        brevity_penalty = 1.0
    else:
        brevity_penalty = math.exp(1 - len(reference) / len(candidate))
    return brevity_penalty * geometric_mean

reference = "the cat is sitting on the mat"

candidates = [
    "the cat is sitting on the mat",
    "the cat is sitting on a mat",
    "on the mat the cat is sitting",
    "the cat",
    "the the the the the the the",
    "a feline rests upon the rug",
]

print(f"{'candidate':<32} {'p1':>5} {'p2':>5} {'BLEU-2':>7} {'BLEU-4':>7}")
for candidate in candidates:
    p1 = modified_precision(candidate.split(), reference.split(), 1)
    p2 = modified_precision(candidate.split(), reference.split(), 2)
    print(f"{candidate:<32} {p1:>5.2f} {p2:>5.2f} {bleu(candidate, reference, 2):>7.2f} {bleu(candidate, reference, 4):>7.2f}")

Go through the rows.

- A perfect copy scores 1.
- Changing one word costs a little at the unigram level and more at higher orders, since every n-gram that includes the changed word is lost.
- The reordered sentence has **all** the right words ($p_1 = 1$). Its score drops because some word pairs are broken. This is how BLEU rewards correct word order without knowing any grammar.
- *"the cat"* has perfect precision and is crushed by the **brevity penalty**.
- *"the the the ..."*: without clipping its unigram precision would be 1. With clipping, only two of its seven words count.
- The last candidate is an acceptable translation in different words, and it scores **zero**.

We can check our implementation against NLTK's.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

candidate = "the cat is sitting on a mat"
print("Ours:", round(bleu(candidate, reference, 4), 4))
print("NLTK:", round(sentence_bleu([reference.split()], candidate.split()), 4))

### What BLEU does and does not tell you
BLEU is fast, cheap and repeatable, and across many sentences it agrees reasonably well with human judgement. That is why it has been the standard yardstick of machine translation for two decades. Its limits follow directly from what it counts.

- It has **no notion of meaning**. Synonyms and paraphrases score nothing, and a fluent sentence that says the opposite of the reference can score well: *"the cat is **not** sitting on the mat"* shares almost every n-gram with it.
- It is meaningful only **over a whole test set**, never for a single sentence, and only when comparing systems on the **same** test set with the same tokenisation.
- A higher BLEU means "closer to these particular references", which is not the same as "a better translation".

> 🧠 Every automatic measure in NLP is a stand-in for the judgement of a person: accuracy, perplexity, BLEU, and the ROUGE score you will meet in Notebook 17. They are indispensable for day-to-day development, and each can be improved without the thing it stands for getting any better. When a result matters, a person should read the output.

### A real translator
The same encoder-decoder design, built from Transformers and trained on millions of sentence pairs, is available as a pretrained model. The cell below downloads one for English to French (about 300 MB) and scores its output against a reference translation with our own `bleu` function.

In [ ]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

english = "The cat is sitting on the mat in front of the door."
french_reference = "Le chat est assis sur le tapis devant la porte ."

machine_output = translator(english)[0]["translation_text"]
print(machine_output)

tokens = machine_output.replace(".", " .").replace("'", "' ")       # separate punctuation, roughly
print("BLEU-2 against our reference:", round(bleu(tokens, french_reference, 2), 2))

Whatever the score, read the French and judge it yourself. If the output is a good translation that differs from our single reference in a word or two, you are looking at the limitation described above.

---
## ✏️ Exercises

### Exercise 1 (Teacher forcing on paper)
The target of a training example is the string `1999-12-25`. Write down, on paper, the decoder's **input** sequence and its **target** sequence, including the special symbols. How long is each? Check with `make_batch`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
sources, decoder_inputs, decoder_targets = make_batch([("25 dec 1999", "1999-12-25")], date_input_characters, date_output_characters)

print("input: ", show(decoder_inputs[0], date_output_characters))
print("target:", show(decoder_targets[0], date_output_characters))
print(len(decoder_inputs[0]), len(decoder_targets[0]))
```

*Both have 11 items: the 10 characters of the date plus one special symbol. The input starts with `<s>`, and the target ends with `</s>`. At every position the target is the item that follows the input.*
</details>

### Exercise 2 (Outside the training data)
Our date model was trained on four formats. Try it on inputs of kinds it has **never seen**: `"3rd of march 2021"`, `"2021 march 3"`, `"march 2021"`, `"31 february 2021"` and `"hello world"`. What does it do? Does it ever say that it does not know?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for text in ["3rd of march 2021", "2021 march 3", "march 2021", "31 february 2021", "hello world"]:
    print(f"{text:<20} -> {translate(date_system, text)}")
```

*The model nearly always answers with something shaped like a date, even for "hello world", where it writes a confident string of digits and dashes. It converts the impossible 31 February without complaint. It cannot say "I do not know": producing a plausible output is the only thing it was trained to do. The larger language models of Module F share this weakness, and there it has a name: hallucination.*
</details>

### Exercise 3 (BLEU by hand)
For the reference `"the quick brown fox jumps"` and the candidate `"the brown fox jumps high"`, compute on paper the unigram precision, the bigram precision, the brevity penalty and BLEU-2. Then check with the functions of Section 6.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
reference_2 = "the quick brown fox jumps"
candidate_2 = "the brown fox jumps high"

# unigrams of the candidate: the, brown, fox, jumps, high -> 4 of 5 are in the reference: 0.8
# bigrams of the candidate: "the brown", "brown fox", "fox jumps", "jumps high" -> 2 of 4: 0.5
# both have 5 words, so the brevity penalty is exp(1 - 5/5) = 1
# BLEU-2 = 1 * sqrt(0.8 * 0.5) = 0.632
print(modified_precision(candidate_2.split(), reference_2.split(), 1))
print(modified_precision(candidate_2.split(), reference_2.split(), 2))
print(round(bleu(candidate_2, reference_2, 2), 3))
```

*The geometric mean of two numbers is the square root of their product.*
</details>

### Exercise 4 (Fooling the metric)
Find the BLEU-2 score, against the reference `"the cat is sitting on the mat"`, of these two candidates: `"the cat is not sitting on the mat"` and `"a cat sits upon a rug"`. Which candidate is the better translation? Which has the higher score?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for candidate in ["the cat is not sitting on the mat", "a cat sits upon a rug"]:
    print(f"{bleu(candidate, reference, 2):.2f}   {candidate}")
```

*The sentence that says the opposite of the reference scores very high, and the faithful paraphrase scores zero. BLEU counts shared word sequences. It knows nothing about meaning.*
</details>

### Exercise 5 (A smaller context vector, a little harder)
The context of our date model has 128 numbers. Train a new date model with `hidden_size=8` for 700 steps, and measure its exact-match accuracy on `test_pairs`. Then look at ten of its mistakes. Which part of the date does it get wrong?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
small_system = train_seq2seq(random_date_pair, steps=700, hidden_size=8, report_every=350)

correct = sum(1 for written, standard in test_pairs if translate(small_system, written) == standard)
print(f"\nExact-match accuracy with 8 hidden units: {correct / len(test_pairs):.1%}\n")

shown = 0
for written, standard in test_pairs:
    answer = translate(small_system, written)
    if answer != standard:
        print(f"{written:<20} -> {answer}   (correct: {standard})")
        shown += 1
        if shown == 10:
            break
```

*With so small a context the model can no longer carry the date across, and its accuracy collapses. Notice that almost all the wrong answers still look like dates. The decoder has learned the format. What it lacks is information about the input. A fluent output is no evidence that the content is correct.*
</details>

## 🔑 Key takeaways

- **Sequence-to-sequence** tasks map an input sequence to an output sequence of different length and order: translation, summarisation, question answering.
- The **encoder** compresses the input into a **context**. The **decoder** is a language model that starts from that context.
- **Teacher forcing** trains the decoder on the correct previous outputs. At **decoding** time it must consume its own.
- A seq2seq model learns from examples what would otherwise take pages of hand-written rules.
- The single fixed-size context is a **bottleneck**: accuracy falls as inputs get longer. **Attention** removes it, and leads to the Transformer.
- **BLEU** scores a translation by clipped n-gram precision against references, with a brevity penalty. It is useful across a test set and blind to meaning.
- A model that always produces well-formed output can be confidently wrong.

---
**Next:** *Notebook 14: Convolutions over Text*, a different way for a neural network to read a sentence: not word by word, but by sliding small pattern detectors along it.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*